## Gold Layer — Step 1: Customer Transaction Summary

#### First, load our Silver tables:

In [0]:
df_customers = spark.table("silver_customers")
df_accounts = spark.table("silver_accounts")
df_transactions = spark.table("silver_transactions")

#### Then create the first Gold dataset:

In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, round
)

gold_customer_summary = (
    df_customers
    .join(
        df_accounts,
        "customer_id",
        "left"
    )
    .join(
        df_transactions,
        "account_id",
        "left"
    )
    .groupBy(
        df_customers.customer_id,
        df_customers.name,
        df_customers.city,
        df_customers.state
    )
    .agg(
        count("transaction_id").alias("total_transactions"),
        round(sum("amount"), 2).alias("total_transaction_amount"),
        round(avg("amount"), 2).alias("average_transaction_amount")
    )
)

display(gold_customer_summary)

customer_id,name,city,state,total_transactions,total_transaction_amount,average_transaction_amount
C0752,Tristan Koshy,Bangalore,Karnataka,14,357413.85,25529.56
C0798,Bina Sarkar,Kochi,Kerala,20,765339.97,38267.0
C0942,Nidhi Mishra,Coimbatore,Tamil Nadu,11,276928.83,25175.35
C0229,Ishita Chatterjee,Pune,Maharashtra,14,392264.61,28018.9
C0234,Baljiwan Gandhi,Delhi,Delhi,15,438266.6,29217.77
C0246,Vyanjana Khosla,Delhi,Delhi,6,187986.76,31331.13
C0385,Oliver Manda,Mumbai,Maharashtra,19,401690.29,21141.59
C0548,Faqid Kuruvilla,Kochi,Kerala,18,775683.42,43093.52
C0693,Jagat Chandra,Pune,Maharashtra,14,423583.92,30255.99
C0665,Vyanjana Deol,Chennai,Tamil Nadu,11,329542.56,29958.41


#### Cell 3 — Save Customer Transaction Summary and Verify

In [0]:
gold_customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_transaction_summary")

display(spark.table("gold_customer_transaction_summary"))

customer_id,name,city,state,total_transactions,total_transaction_amount,average_transaction_amount
C0752,Tristan Koshy,Bangalore,Karnataka,14,357413.85,25529.56
C0798,Bina Sarkar,Kochi,Kerala,20,765339.97,38267.0
C0942,Nidhi Mishra,Coimbatore,Tamil Nadu,11,276928.83,25175.35
C0229,Ishita Chatterjee,Pune,Maharashtra,14,392264.61,28018.9
C0234,Baljiwan Gandhi,Delhi,Delhi,15,438266.6,29217.77
C0246,Vyanjana Khosla,Delhi,Delhi,6,187986.76,31331.13
C0385,Oliver Manda,Mumbai,Maharashtra,19,401690.29,21141.59
C0548,Faqid Kuruvilla,Kochi,Kerala,18,775683.42,43093.52
C0693,Jagat Chandra,Pune,Maharashtra,14,423583.92,30255.99
C0665,Vyanjana Deol,Chennai,Tamil Nadu,11,329542.56,29958.41


### Daily Transaction Summary.
#### Cell 5 — Create Gold Daily Summary

In [0]:
from pyspark.sql.functions import count, sum, avg, round

gold_daily_summary = (
    df_transactions
    .groupBy("transaction_date")
    .agg(
        count("transaction_id").alias("total_transactions"),
        round(sum("amount"), 2).alias("total_amount"),
        round(avg("amount"), 2).alias("average_amount")
    )
    .orderBy("transaction_date")
)

display(gold_daily_summary)

transaction_date,total_transactions,total_amount,average_amount
2026-01-01,42,1045245.03,24886.79
2026-01-02,42,1263080.21,30073.34
2026-01-03,41,1000845.18,24410.86
2026-01-04,31,713150.99,23004.87
2026-01-05,38,944397.12,24852.56
2026-01-06,31,695067.46,22421.53
2026-01-07,34,732769.53,21552.05
2026-01-08,27,862364.82,31939.44
2026-01-09,34,917859.27,26995.86
2026-01-10,41,952139.76,23222.92


#### Cell 6 — Save it

In [0]:
gold_daily_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_transaction_summary")

### Account Transaction Summary.

#### Cell 7 — Create Gold Account Summary

In [0]:
from pyspark.sql.functions import count, sum, avg, round

gold_account_summary = (
    df_accounts
    .join(
        df_transactions,
        "account_id",
        "left"
    )
    .groupBy(
        "account_id",
        "customer_id",
        "account_type"
    )
    .agg(
        count("transaction_id").alias("total_transactions"),
        round(sum("amount"), 2).alias("total_transaction_amount"),
        round(avg("amount"), 2).alias("average_transaction_amount")
    )
)

display(gold_account_summary)

account_id,customer_id,account_type,total_transactions,total_transaction_amount,average_transaction_amount
A00009,C0931,Current,9,202678.9,22519.88
A00509,C0613,Current,7,159890.22,22841.46
A00590,C0500,Current,10,114013.27,11401.33
A00698,C0268,Current,13,325555.72,25042.75
A00761,C0383,Current,7,106614.96,15230.71
A01063,C0844,Savings,7,210069.94,30009.99
A00805,C0391,Current,6,204793.95,34132.33
A01123,C0908,Savings,9,195186.42,21687.38
A00057,C0783,Current,13,531476.29,40882.79
A00211,C0587,Current,9,276890.49,30765.61


#### Cell 8 — Save and Verify

In [0]:
gold_account_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_account_transaction_summary")

    
display(spark.table("gold_account_transaction_summary"))

account_id,customer_id,account_type,total_transactions,total_transaction_amount,average_transaction_amount
A00009,C0931,Current,9,202678.9,22519.88
A00509,C0613,Current,7,159890.22,22841.46
A00590,C0500,Current,10,114013.27,11401.33
A00698,C0268,Current,13,325555.72,25042.75
A00761,C0383,Current,7,106614.96,15230.71
A01063,C0844,Savings,7,210069.94,30009.99
A00805,C0391,Current,6,204793.95,34132.33
A01123,C0908,Savings,9,195186.42,21687.38
A00057,C0783,Current,13,531476.29,40882.79
A00211,C0587,Current,9,276890.49,30765.61


### final Gold KPI table: transaction type + channel analysis.
#### Cell 10 — Transaction Type & Channel Summary

In [0]:
from pyspark.sql.functions import count, sum, avg, round

gold_transaction_analysis = (
    df_transactions
    .groupBy(
        "transaction_type",
        "channel"
    )
    .agg(
        count("transaction_id").alias("total_transactions"),
        round(sum("amount"), 2).alias("total_amount"),
        round(avg("amount"), 2).alias("average_amount")
    )
    .orderBy("transaction_type", "channel")
)

display(gold_transaction_analysis)

transaction_type,channel,total_transactions,total_amount,average_amount
Credit,ATM,836,2.246271326E7,26869.27
Credit,Branch,853,2.332403163E7,27343.53
Credit,IMPS,839,2.403988584E7,28653.02
Credit,NEFT,863,2.332611624E7,27029.1
Credit,Online,830,2.159110252E7,26013.38
Credit,UPI,821,2.330609105E7,28387.44
Debit,ATM,840,2.411000248E7,28702.38
Debit,Branch,817,2.298193767E7,28129.67
Debit,IMPS,848,2.194258002E7,25875.68
Debit,NEFT,864,2.364318024E7,27364.79


#### Cell 11 — Save Gold table and Verify

In [0]:
gold_transaction_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_transaction_analysis")

display(spark.table("gold_transaction_analysis"))

transaction_type,channel,total_transactions,total_amount,average_amount
Credit,ATM,836,2.246271326E7,26869.27
Credit,Branch,853,2.332403163E7,27343.53
Credit,IMPS,839,2.403988584E7,28653.02
Credit,NEFT,863,2.332611624E7,27029.1
Credit,Online,830,2.159110252E7,26013.38
Credit,UPI,821,2.330609105E7,28387.44
Debit,ATM,840,2.411000248E7,28702.38
Debit,Branch,817,2.298193767E7,28129.67
Debit,IMPS,848,2.194258002E7,25875.68
Debit,NEFT,864,2.364318024E7,27364.79
